In [ ]:
import random
import copy


def random_matrix():
    nums = [0, 1, 2, 3, 4, 5, 6, 7, 8]
    matrix = [[0 for _ in range(3)] for _ in range(3)]
    for i in range(3):
        for j in range(3):
            num = random.choice(nums)
            matrix[i][j] = num
            nums.remove(num)
    return matrix

def get_actions(i, j):
    actions = []
    if i > 0: actions.append("up")
    if i < 2: actions.append("down")
    if j > 0: actions.append("left")
    if j < 2: actions.append("right")
    return actions

def percept(matrix):
    return [row for row in matrix]


def update_state(state, action, new_percept, world_model):
    if action is None:
        return new_percept
    predicted = world_model(state, action)
    if predicted == new_percept:
        return predicted
    else:
        return new_percept


def rule_match(state, rules):
    return rules(state)

def rules(board):
    i0, j0 = next((i, j) for i in range(3) for j in range(3) if board[i][j] == 0)
    candidates = get_actions(i0, j0)
    return candidates


def puzzle_8(new_percept, history):
    persistent.state = update_state(
        persistent.state,
        persistent.action,
        new_percept,
        persistent.model
    )

    rule = rule_match(persistent.state, rules)

    for action in rule:
        predicted_board = persistent.model(persistent.state, action)
        predicted_tuple = tuple(tuple(row) for row in predicted_board)
        if predicted_tuple not in history:
            persistent.action = action
            return action

    action = random.choice(rule)
    persistent.action = action
    return action



class persistent:
    state  = None
    action = None

    def model(board, action):
        moves = {
            "up": (-1, 0),
            "down": (+1, 0),
            "left": (0, -1),
            "right": (0, +1)
        }
        i0, j0 = next((i, j) for i in range(3) for j in range(3) if board[i][j] == 0)
        move = moves[action]
        i1, j1 = i0 + move[0], j0 + move[1]

        new_board = copy.deepcopy(board)
        new_board[i0][j0], new_board[i1][j1] = new_board[i1][j1], new_board[i0][j0]
        return new_board

def actuator(matrix, action):
    moves = {
        "up":    (-1, 0),
        "down":  (+1, 0),
        "left":  (0, -1),
        "right": (0, +1)
    }
    i0, j0 = next(
        (i, j)
        for i in range(3)
        for j in range(3)
        if matrix[i][j] == 0
    )
    move = moves[action]
    i1, j1 = i0 + move[0], j0 + move[1]
    matrix[i0][j0], matrix[i1][j1] = matrix[i1][j1], matrix[i0][j0]
    return matrix


if __name__ == "__main__":
    matrix = random_matrix()
    GOAL = [
        [1, 2, 3],
        [4, 5, 6],
        [7, 8, 0]
    ]
    for row in matrix:
        print(row)
    print()

    history = set()
    path = []

    while True:
        state_tuple = tuple(tuple(row) for row in matrix)

        if matrix == GOAL:
            print(f"Đã giải được puzzle sau {len(path)} bước!")
            break

        if state_tuple in history:
            print("Không thể giải puzzle (đã thăm toàn bộ nước đi khả dụng).")
            break

        history.add(state_tuple)

        observation = percept(matrix)
        action = puzzle_8(observation, history)
        path.append(action)
        matrix = actuator(matrix, action)

    print(" -> ".join(path))
    for row in matrix:
        print(row)

[1, 0, 8]
[3, 2, 7]
[6, 4, 5]

Không thể giải puzzle (đã thăm toàn bộ nước đi khả dụng).
down -> down -> left -> up -> up -> right -> down -> down -> left -> up -> up -> right -> down -> down -> left -> up -> up -> right -> down -> down -> left -> up -> up -> right -> down -> down -> left -> up -> up -> down
[3, 1, 8]
[0, 2, 7]
[6, 4, 5]


Ở Simple-Based thì AI Agent chỉ đi ngẫu nhiên một cách mù quáng còn ở Model-Based thì AI Agent đã có lưu vết trạng thái cũ và dựa vào trạng thái cũ đó để đưa ra nước đi mới sao cho hợp lý hơn hạn chế đi vào ngõ cụt.